In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import scipy
from scipy.stats import multivariate_normal

print("Installations Working!")

Installations Working!


# Vechicle Position Problem


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# ---------- Settings ----------
sigma_r = 0.3                 # measurement noise std (same for all landmarks)
sigma_x = sigma_y = 0.25      # prior std: how strongly we believe the vehicle is near the origin

# ---------- True vehicle position: uniform inside the unit circle ----------
radius = np.sqrt(rng.random())          # sqrt makes it uniform over the AREA, not bunched at the center
angle = 2 * np.pi * rng.random()
true_pos = np.array([radius * np.cos(angle), radius * np.sin(angle)])

# ---------- Helper functions ----------
def make_landmarks(K):
    """K evenly spaced points on the unit circle."""
    angles = 2 * np.pi * np.arange(K) / K
    return np.column_stack([np.cos(angles), np.sin(angles)])

def measure_ranges(true_pos, landmarks, sigma, rng):
    """r_i = true distance + Gaussian noise; redraw any negative measurement."""
    d_true = np.linalg.norm(landmarks - true_pos, axis=1)
    r = np.empty(len(landmarks))
    for i, d in enumerate(d_true):
        r_i = -1
        while r_i < 0:
            r_i = d + sigma * rng.standard_normal()
        r[i] = r_i
    return r

def map_objective(X, Y, landmarks, r, sigma_r, sigma_x, sigma_y):
    """Simplified MAP objective, evaluated at every (x, y) on the grid at once."""
    J = X**2 / sigma_x**2 + Y**2 / sigma_y**2                    # prior term
    for (x_i, y_i), r_i in zip(landmarks, r):
        d_i = np.sqrt((X - x_i)**2 + (Y - y_i)**2)                # candidate-to-landmark distance
        J += (r_i - d_i)**2 / sigma_r**2                          # measurement term
    return J

# ---------- Evaluation grid ----------
g = np.linspace(-2, 2, 401)
X, Y = np.meshgrid(g, g)

# ---------- Compute everything first (needed for shared contour levels) ----------
results = {}
for K in [1, 2, 3, 4]:
    L = make_landmarks(K)
    r = measure_ranges(true_pos, L, sigma_r, rng)
    J = map_objective(X, Y, L, r, sigma_r, sigma_x, sigma_y)
    idx = np.unravel_index(np.argmin(J), J.shape)
    map_est = np.array([X[idx], Y[idx]])                          # grid-search MAP estimate
    results[K] = (L, r, J, map_est)

# Same contour levels for every K
all_J = np.concatenate([res[2].ravel() for res in results.values()])
levels = np.geomspace(max(all_J.min(), 0.1), np.percentile(all_J, 95), 20)

# ---------- Plot ----------
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for ax, K in zip(axes.ravel(), [1, 2, 3, 4]):
    L, r, J, map_est = results[K]
    cs = ax.contour(X, Y, J, levels=levels, cmap="viridis")
    ax.plot(*true_pos, 'r+', markersize=14, mew=2.5, label="True position")
    ax.plot(L[:, 0], L[:, 1], 'ko', markersize=8, mfc='none', mew=2, label="Landmarks")
    ax.plot(*map_est, 'm*', markersize=13, label="MAP estimate")
    ax.set_title(f"K = {K}   |   MAP error = {np.linalg.norm(map_est - true_pos):.3f}")
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect('equal')
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)

fig.colorbar(cs, ax=axes, shrink=0.6, label="MAP objective value")
fig.suptitle(f"MAP Objective Contours  (σ_r = {sigma_r}, σ_x = σ_y = {sigma_x})", fontsize=14)
plt.show()

print("True position:", true_pos)
for K, (L, r, J, map_est) in results.items():
    print(f"K={K}: ranges = {np.round(r, 3)}, MAP estimate = {np.round(map_est, 3)}")